# Expense Approval Agent | Human-in-the-Loop (HITL)

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from typing import TypedDict, Literal
from typing_extensions import NotRequired
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke
import json
import re

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

class ExpenseState(TypedDict):
    request: str
    amount: NotRequired[float]
    category: NotRequired[str]
    justification: NotRequired[str]
    human_decision: NotRequired[str]
    result: NotRequired[str]

APPROVAL_THRESHOLD = 500.0  # Expenses above this require human approval

In [4]:
def analyze_expense(state: ExpenseState) -> Command[Literal["request_human_approval", "auto_approve"]]:
    """Parse and categorize the expense request, then route based on amount threshold."""
    response = model.invoke(
        f"Analyze this expense request and extract:\n"
        f"1. Amount (number only)\n"
        f"2. Category (travel, software, equipment, meals, other)\n"
        f"3. Business justification (one sentence)\n\n"
        f"Request: {state['request']}\n\n"
        f'Return JSON: {{"amount": 123.45, "category": "...", "justification": "..."}}'
    )
    cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", response.content).strip()
    parsed = json.loads(cleaned)
    amount = float(parsed.get("amount", 0))
    category = str(parsed.get("category", "other"))
    justification = str(parsed.get("justification", ""))
    update = {"amount": amount, "category": category, "justification": justification}
    if amount > APPROVAL_THRESHOLD:
        return Command(goto="request_human_approval", update=update)
    return Command(goto="auto_approve", update=update)

In [5]:
def request_human_approval(state: ExpenseState) -> Command[Literal["process_expense", "deny_expense"]]:
    """Pause execution and wait for human decision using LangGraph interrupt."""
    decision = interrupt(
        f"APPROVAL REQUIRED:\n"
        f"  Amount: ${state['amount']:.2f}\n"
        f"  Category: {state['category']}\n"
        f"  Justification: {state['justification']}\n\n"
        f"Reply 'approved' or 'denied: <reason>'"
    )
    if decision.strip().lower().startswith("approved"):
        return Command(goto="process_expense", update={"human_decision": decision})
    return Command(goto="deny_expense", update={"human_decision": decision})

In [6]:
def auto_approve(state: ExpenseState) -> dict:
    return {"result": f"AUTO-APPROVED: ${state['amount']:.2f} for {state['category']}. {state['justification']}"}

def process_expense(state: ExpenseState) -> dict:
    return {"result": f"HUMAN-APPROVED: ${state['amount']:.2f} for {state['category']}. {state['justification']}"}

def deny_expense(state: ExpenseState) -> dict:
    reason = state.get("human_decision", "No reason given")
    return {"result": f"DENIED: ${state['amount']:.2f}. Decision: {reason}"}

In [7]:
# Build graph
graph = StateGraph(ExpenseState)
graph.add_node("analyze", analyze_expense)
graph.add_node("request_human_approval", request_human_approval)
graph.add_node("auto_approve", auto_approve)
graph.add_node("process_expense", process_expense)
graph.add_node("deny_expense", deny_expense)

graph.add_edge(START, "analyze")
# No add_conditional_edges needed -- analyze and request_human_approval return Command to route directly
graph.add_edge("auto_approve", END)
graph.add_edge("process_expense", END)
graph.add_edge("deny_expense", END)

checkpointer = InMemorySaver()
app = graph.compile(checkpointer=checkpointer)

In [8]:
# Plot the workflow
plot_mermaid(app)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	analyze(analyze)
	request_human_approval(request_human_approval)
	auto_approve(auto_approve)
	process_expense(process_expense)
	deny_expense(deny_expense)
	__end__([<p>__end__</p>]):::last
	__start__ --> analyze;
	analyze -.-> auto_approve;
	analyze -.-> request_human_approval;
	request_human_approval -.-> deny_expense;
	request_human_approval -.-> process_expense;
	auto_approve --> __end__;
	deny_expense --> __end__;
	process_expense --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [9]:
# --- Auto-approve path: small expense (no human needed) ---
config_small = {"configurable": {"thread_id": "expense-small-001"}}
result = app.invoke(
    {"request": "Need $45 for team lunch with client"},
    config=config_small
)
print(result["result"])

AUTO-APPROVED: $45.00 for meals. Team lunch with client


In [10]:
# --- HITL path: large expense (pauses for human approval) ---
config_large = {"configurable": {"thread_id": "expense-large-001"}}

# First invocation -- pauses at interrupt if amount > $500
result = app.invoke(
    {"request": "Need $1,200 for a conference flight to San Francisco for the AI summit"},
    config=config_large
)
print("Paused for approval. State:", result.get("result", "Awaiting human decision..."))

Paused for approval. State: Awaiting human decision...


In [11]:
# Simulate human approval by resuming with a Command
result = app.invoke(Command(resume="approved"), config=config_large)
print(result["result"])

HUMAN-APPROVED: $1200.00 for travel. Need $1,200 for a conference flight to San Francisco for the AI summit.


In [12]:
# Streaming (auto-approve path -- stream_invoke works on non-interrupt executions)

stream_invoke(
    app,
    {"request": "Need $80 for team dinner after product launch"},
    config={"configurable": {"thread_id": "expense-stream-001"}}
)


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'request': 'Need $80 for team dinner after product launch',
 'amount': 80.0,
 'category': 'meals',
 'justification': 'Team dinner after product launch.',
 'result': 'AUTO-APPROVED: $80.00 for meals. Team dinner after product launch.'}